[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/pyPetrograph/blob/main/pyPetrograph_1_rgb_porosity.ipynb)


# pyPetrograph — RGB porosity demo (v0.0.7)

Label pores and grains on three PPL test images, train a LightGBM pixel classifier, then report **porosity %** with uncertainty.

Example labels are already in `outputs/labels/` (2 classes: pores, grains; cements from the old 3-class set were folded into grains. Notebook 3 keeps its own object-level names, including cements). The label window opens by default (`RELABEL = True`). Set `RELABEL = False` to skip it.

No TensorFlow, U-Net, or SAM. Run cells **in order**. Outputs go to `outputs/` (`labels/`, `predictions/`, `models/`).


## Setup — check packages

In [ ]:
from pathlib import Path
import sys
ROOT = Path(".")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

from pyPetrograph import check_and_install_packages
_ = check_and_install_packages(install=True)


## Setup — imports

In [ ]:
%matplotlib inline

import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 150
mpl.rcParams["savefig.dpi"] = 150

from pathlib import Path

from pyPetrograph import (
    Session,
    set_output_dir,
    stem_paths,
    select_images,
    launch_app,
    print_label_summary,
    print_train_metrics,
    run_feature_preview,
    run_train_cell,
    run_predict_cell,
)


## Settings

Two classes: **pores** and **grains**. Queue is the three PPL tiffs in `example_data/`. `select_images` is optional if you want a different set.


In [ ]:
from pyPetrograph import set_output_dir
set_output_dir("outputs")

session = Session()
session.class_names = {
    1: "pores",
    2: "grains",
}
session.remember_settings_classes()
session.current_class = 1

IMAGE_DIR = Path("example_data")
session.image_dir = IMAGE_DIR

PPLS = [
    IMAGE_DIR / "AV03_ppl_10x_pic03t3.tif",
    IMAGE_DIR / "AV11_ppl_10x_pic04t4.tif",
    IMAGE_DIR / "AV12_ppl_10x_pic03t3.tif",
]
session.set_image_queue(PPLS)

session.preview_dpi = 150
session.figure_dpi = 150
session.draw_mode = "wand"
session.wand_threshold = 70
session.method = "lightgbm"
session.n_jobs = 6

RELABEL = True  # False skips the wand/polygon window

print("Classes:", session.class_names)
print("queue:", [p.name for p in session.image_queue])
print("outputs:", Path("outputs").resolve())


## Select images (optional)

Skip this cell — the Settings cell already queued the three PPL test images.


In [ ]:
# session = select_images(session)
# session = select_images(session, append=True)
print("queue:", [p.name for p in session.image_queue])


## Interactive labeling

Opens the wand/polygon Qt window. Example polygons are already in `outputs/labels/`. AV12 has no example labels (train skips it; predict still runs). Set `RELABEL = False` in Settings to skip.


In [ ]:
if RELABEL:
    session = launch_app(session)
else:
    print("RELABEL is False — using outputs/labels/")


## Label summary

In [ ]:
print_label_summary(session)

## Features

The classifier looks at color plus two texture maps:

- **local std** — how much the color changes in a small neighborhood (smooth vs speckled)
- **local gradient** — edges (grain boundaries)

Training still uses the full-resolution image. This cell only shows a downsampled preview.


In [ ]:
session.feature_toggles = {
    "r": True,
    "g": True,
    "b": True,
    "local_std": True,
    "local_grad": True,
}
run_feature_preview(session)


## Train

Fits one LightGBM model on all labeled pixels from AV03 and AV11. AV12 has no labels, so it is skipped. Writes `outputs/models/universal_model.joblib`.


In [ ]:
train_results = run_train_cell(
    session,
    mode="retrain",
    scope="universal",
    prompt_load=False,
)


## Cross-validation metrics

How well the model labels pixels it did not train on (a 5-fold split of the labeled pixels). This is not a test on a new thin section.


In [ ]:
print_train_metrics(train_results, session=session)

## Predict

Class map for every queued image, including AV12. Files go to `outputs/predictions/`.


In [ ]:
predict_results = run_predict_cell(
    session,
    model_source="universal",
)


## Porosity %

**porosity_pct** is the fraction of pixels called pores.

**plus** / **minus** are a confidence range, not a lab error bar:

- **minus** — drop pore pixels the model is unsure about (probability below 0.8)
- **plus** — also count pixels that look a bit like pores

**mean ± sd** is the average of the three images, and how much those three values differ from each other.


In [ ]:
import pandas as pd

rows = []
for path in session.image_queue:
    csv = stem_paths(path).fractions_csv
    if not csv.exists():
        print("missing", csv.name)
        continue
    df = pd.read_csv(csv)
    pore = df.loc[df["name"].str.lower() == "pores"]
    if pore.empty:
        print(path.stem, "no pores row")
        continue
    r = pore.iloc[0]
    rows.append({
        "image": path.stem,
        "porosity_pct": float(r["area_pct"]),
        "plus": float(r.get("area_pct_plus", 0)),
        "minus": float(r.get("area_pct_minus", 0)),
    })
    print(f"{path.stem}: {r['area_pct']:.2f}% +{r.get('area_pct_plus', 0):.2f}%/−{r.get('area_pct_minus', 0):.2f}%")

tab = pd.DataFrame(rows)
if len(tab):
    sd = float(tab["porosity_pct"].std(ddof=1)) if len(tab) > 1 else 0.0
    print(f"mean ± sd over {len(tab)} image(s): {tab['porosity_pct'].mean():.2f}% ± {sd:.2f}%")
tab
